[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hany73-max/PDE-Rover-Engine/blob/main/labs_ar/Lab_6_Mars_Rover_Capstone_AR.ipynb)


In [ ]:
# 🚀 تهيئة بيئة Google Colab
# إذا كنت تشغل هذا النوت بوك على Google Colab، شغل هذه الخلية لتحميل ملفات المستودع والاختبارات تلقائياً:
import os, sys
try:
    if 'google.colab' in str(get_ipython()):
        if not os.path.exists('PDE-Rover-Engine'):
            import subprocess
            subprocess.run(['git', 'clone', 'https://github.com/hany73-max/PDE-Rover-Engine.git'], check=True)
        else:
            import subprocess
            subprocess.run(['git', '-C', 'PDE-Rover-Engine', 'pull', '--ff-only'], check=False)
        os.chdir('PDE-Rover-Engine')
        if '.' not in sys.path:
            sys.path.insert(0, '.')
        print('[OK] تم تجهيز بيئة Google Colab بنجاح والاتصال بمستودع المشروع!')
except NameError:
    pass


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:calc(100vw - 140px); max-width:calc(100vw - 140px); min-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">
<style>.markdown, .markdown-body, :host { max-width: 100% !important; width: 100% !important; }</style>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

# ‏🚀 اللاب 6: المشروع النهائي الشامل — الفروق المحدودة، خوارزمية SOR، والملاحة الذاتية على المريخ

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

### ‏<i>غرفة التحكم: دمج المعادلات التفاضلية الجزئية في محرك ملاحة المركبة الكامل</i>

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏<b>المقرر:</b> المعادلات التفاضلية الجزئية وتطبيقاتها في الملاحة الفضائية (Applied PDE & Planetary Robotics)

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏<b>المرجع:</b> التكامل التطبيقي الشامل لما درسناه في اللابات 1–5 مع الكود الفعلي في <a href="rover_path_finder/solvers.py">‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">rover&#95;path&#95;finder/solvers.py</code>‏</a> و <a href="rover_path_finder/planners.py">‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">rover&#95;path&#95;finder/planners.py</code>‏</a>

</div>

</div>


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:calc(100vw - 140px); max-width:calc(100vw - 140px); min-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">
<style>.markdown, .markdown-body, :host { max-width: 100% !important; width: 100% !important; }</style>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

### ‏🗺️ خريطة الكورس الكاملة: كيف تجمّعت قطع الليغو الخمس (اللابات 1–5) لبناء محرك الملاحة؟

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; background-color:rgba(128,128,128,0.12); padding:10px 14px; border-radius:6px; overflow-x:auto; margin:10px 0; font-family:monospace; white-space:pre;">‏[اللاب 1: حساب الميول slope_map] ──► [اللاب 2: خطوط لاجرانج descend] ──► [اللاب 3: مسافة الأمان dilate]
                                                                                      │
📍 [اللاب 6: الحل العددي SOR + Neumann وإطلاق المركبة] ◄── [اللاب 5: مشكلة الممرات الضيقة] ◄── [اللاب 4: معادلة لابلاس _avg]</div>

<div dir="rtl" style="direction:rtl; text-align:right; overflow-x:auto; width:100%; margin:14px 0;">
<table dir="rtl" style="display:table; width:100%; border-collapse:collapse; direction:rtl; text-align:right;">
<thead>
<tr>
<th dir="rtl" align="right" style="direction:rtl; text-align:right; padding:9px 12px; border:1px solid rgba(128,128,128,0.4); background-color:rgba(128,128,128,0.16); font-weight:bold; line-height:1.75; vertical-align:middle;">

‏قطعة البناء (اللاب)

</th>
<th dir="rtl" align="right" style="direction:rtl; text-align:right; padding:9px 12px; border:1px solid rgba(128,128,128,0.4); background-color:rgba(128,128,128,0.16); font-weight:bold; line-height:1.75; vertical-align:middle;">

‏القانون الرياضي في مذكرة الـ PDF (ص 1–62)

</th>
<th dir="rtl" align="right" style="direction:rtl; text-align:right; padding:9px 12px; border:1px solid rgba(128,128,128,0.4); background-color:rgba(128,128,128,0.16); font-weight:bold; line-height:1.75; vertical-align:middle;">

‏المهمة الهندسية التي يحلها داخل مركبة المريخ

</th>
<th dir="rtl" align="right" style="direction:rtl; text-align:right; padding:9px 12px; border:1px solid rgba(128,128,128,0.4); background-color:rgba(128,128,128,0.16); font-weight:bold; line-height:1.75; vertical-align:middle;">

‏الدالة الفعلية في المشروع (‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">rover&#95;path&#95;finder/</code>‏)

</th>
</tr>
</thead>
<tbody>
<tr>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏<b>🧱 القطعة 1 (اللاب 1)</b>

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${p = \frac{\partial z}{\partial x},\; q = \frac{\partial z}{\partial y}}$</span>‏ وحذف الثوابت الاختيارية <i>(PDF ص 2–11)</i>

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏قياس ميل التضاريس ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${\Vert \nabla z\Vert  = \sqrt{p^2+q^2}}$</span>‏ واكتشاف المنحدرات الخطرة (‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">&gt; 0.30</code>‏)

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏<a href="rover_path_finder/terrain.py">‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">terrain.slope&#95;map(elev)</code>‏</a>

</td>
</tr>
<tr>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏<b>🧱 القطعة 2 (اللاب 3)</b>

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏المعادلات غير الخطية ومعادلة المسافة ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${p^2 + q^2 = 1}$</span>‏ <i>(PDF ص 17–27)</i>

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏توسيع كل صخرة وحفرة بمقدار نصف قطر جسم المركبة ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${R}$</span>‏ لمنع اصطدام العجلات

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏<a href="rover_path_finder/terrain.py">‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">terrain.dilate(raw, rover&#95;radius)</code>‏</a>

</td>
</tr>
<tr>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏<b>🧱 القطعة 3 (اللاب 4)</b>

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏معادلة لابلاس ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${(D_x^2 + D_y^2)\phi = 0}$</span>‏ والجذور المركبة ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${m = \pm i}$</span>‏ <i>(PDF ص 28–51)</i>

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏بناء سطح جهد توافقي أملس يستحيل رياضياً أن يحتوي على أي قاع محلي يحبس المركبة

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏<a href="rover_path_finder/solvers.py">‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">solvers.&#95;avg(phi)</code>‏</a>

</td>
</tr>
<tr>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏<b>🧱 القطعة 4 (اللاب 5)</b>

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏العامل الخطي ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${(aD_x+bD_y+c)z=0 \implies e^{-(c/a)x}}$</span>‏ <i>(PDF ص 52–62)</i>

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏تفسير تلاشي الجهد ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${e^{-\pi x/w}}$</span>‏ في الممرات الضيقة واستبدال ديريكليه بشرط نيومان العازل

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏<a href="rover_path_finder/planners.py">‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">planners.neumann&#95;potential()</code>‏</a>

</td>
</tr>
<tr>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏<b>🧱 القطعة 5 (اللاب 2)</b>

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏معادلة لاجرانج للخصائص ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${\frac{dx}{P} = \frac{dy}{Q} = \frac{dz}{R}}$</span>‏ <i>(PDF ص 12–17)</i>

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏تتبع خط الانسياب الأشد انحداراً ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${(-\phi_x, -\phi_y)}$</span>‏ لقيادة المركبة من البداية للهدف

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏<a href="rover_path_finder/planners.py">‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">planners.descend(phi, hazard, start, goal)</code>‏</a>

</td>
</tr>
</tbody>
</table>
</div>

</div>


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:calc(100vw - 140px); max-width:calc(100vw - 140px); min-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">
<style>.markdown, .markdown-body, :host { max-width: 100% !important; width: 100% !important; }</style>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

### ‏🧭 حجر رشيد (The Rosetta Stone): الربط المباشر بين المعادلات وملفات المشروع

</div>

<div dir="rtl" style="direction:rtl; text-align:right; overflow-x:auto; width:100%; margin:14px 0;">
<table dir="rtl" style="display:table; width:100%; border-collapse:collapse; direction:rtl; text-align:right;">
<thead>
<tr>
<th dir="rtl" align="right" style="direction:rtl; text-align:right; padding:9px 12px; border:1px solid rgba(128,128,128,0.4); background-color:rgba(128,128,128,0.16); font-weight:bold; line-height:1.75; vertical-align:middle;">

‏1. المفهوم الرياضي

</th>
<th dir="rtl" align="right" style="direction:rtl; text-align:right; padding:9px 12px; border:1px solid rgba(128,128,128,0.4); background-color:rgba(128,128,128,0.16); font-weight:bold; line-height:1.75; vertical-align:middle;">

‏2. المعنى الفيزيائي والهندسي

</th>
<th dir="rtl" align="right" style="direction:rtl; text-align:right; padding:9px 12px; border:1px solid rgba(128,128,128,0.4); background-color:rgba(128,128,128,0.16); font-weight:bold; line-height:1.75; vertical-align:middle;">

‏3. الكود الفعلي في ‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">rover&#95;path&#95;finder/</code>‏

</th>
</tr>
</thead>
<tbody>
<tr>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${\nabla^2\phi = \frac{\partial^2\phi}{\partial x^2} + \frac{\partial^2\phi}{\partial y^2} = 0}$</span>‏

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏<b>معادلة لابلاس (من اللاب 4):</b> حقل جهد توافقي خالٍ تماماً من القيعان المحلية

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏المعادلة الحاكمة في <a href="rover_path_finder/solvers.py">‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">solvers.py</code>‏</a>

</td>
</tr>
<tr>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${\phi_{i,j} = \frac{1}{4}(\phi_{i+1,j} + \phi_{i-1,j} + \phi_{i,j+1} + \phi_{i,j-1})}$</span>‏

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏<b>قالب النقاط الخمس (5-Point Stencil):</b> تقريب المشتقات الثانية على شبكة بكسل

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏الدالة <a href="rover_path_finder/solvers.py">‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">&#95;avg(phi)</code>‏</a> في ‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">solvers.py</code>‏

</td>
</tr>
<tr>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${\frac{\partial\phi}{\partial n} = 0}$</span>‏ على حدود العوائق

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏<b>شرط نيومان العازل (من اللاب 5):</b> الجهد ينساب حول الصخور ولا يتلاشى في الممرات الضيقة

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏قسمة مجموع الجيران الحرة على عددها الفعلي ‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">cnt</code>‏ في <a href="rover_path_finder/solvers.py">‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">sor&#95;neumann</code>‏</a>

</td>
</tr>
<tr>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${\omega_{\text{opt}} = \frac{2}{1 + \sin(\pi/N)}}$</span>‏

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏<b>معامل الاسترخاء الفائق المتتالي (SOR):</b> تسريع تقارب الشبكة عشرات المرات

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏الدالة <a href="rover_path_finder/solvers.py">‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">optimal&#95;omega(N)</code>‏</a> في ‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">solvers.py</code>‏

</td>
</tr>
<tr>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${\frac{dx}{-\phi_x} = \frac{dy}{-\phi_y}}$</span>‏

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏<b>معادلة لاجرانج لخطوط الانسياب (من اللاب 2):</b> تتبع المسار من البداية للهدف

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏الدالة <a href="rover_path_finder/planners.py">‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">descend(phi, hazard, start, goal)</code>‏</a>

</td>
</tr>
</tbody>
</table>
</div>

</div>


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:calc(100vw - 140px); max-width:calc(100vw - 140px); min-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">
<style>.markdown, .markdown-body, :host { max-width: 100% !important; width: 100% !important; }</style>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

## ‏1. لماذا ننتقل من الحل التحليلي بالورقة والقلم إلى الحل العددي على الشبكة؟

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏في اللابات 1 إلى 5، حللنا المعادلات التفاضلية الجزئية تحليلياً بالورقة والقلم على نطاقات هندسية بسيطة.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏ولكن في مشروع مركبة المريخ لدينا:

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:14px; line-height:1.85;">

‏• بيئة الاختبار القياسية ‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">make&#95;terrain(N=100)</code>‏ تتكون من شبكة <b>‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${100 \times 100 = 10,000}$</span>‏ عقدة</b>، وتضاريس وادي كاسي المريخية الحقيقية (‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">kasei.IMG</code>‏ في ‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">main.py</code>‏) تصل إلى <b>‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${160 \times 160 = 25,600}$</span>‏ عقدة</b> مع آلاف الصخور والحفر غير المنتظمة!

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:14px; line-height:1.85;">

‏• يستحيل كتابة دالة تحليلية مغلقة واحدة تحقق شرط العزل ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\frac{\partial\phi}{\partial n} = 0}$</span>‏ على حدود آلاف الصخور المتعرجة.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:14px; line-height:1.85;">

‏• لذلك نحول معادلة لابلاس المستمرة ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\nabla^2\phi = 0}$</span>‏ إلى نظام معادلات جبرية على الشبكة باستخدام <b>طريقة الفروق المحدودة (Finite Difference Method)</b> ونحله تكرارياً بسرعة فائقة باستخدام <b>خوارزمية Red-Black SOR</b>.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; border-right:4px solid #f59e0b; background-color:rgba(245,158,11,0.08); padding:10px 14px; margin:12px 0; border-radius:4px; line-height:1.85;">

‏📌 <b>ملاحظة منهجية صادقة:</b> دراسة معادلة لابلاس التحليلية ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(D_x^2 + D_y^2)z = 0}$</span>‏ موجودة في المذكرة (ص 39 / PDF ص 33)، أما <b>تقطيع الشبكة (5-Point Stencil) ومبدأ القيمة العظمى (Maximum Principle) وخوارزمية SOR</b> فهي الإثراء العددي والهندسي الذي يربط مذكرة المحاضرات بمحرك البرمجيات الحقيقي في <a href="rover_path_finder/solvers.py">‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">rover&#95;path&#95;finder/solvers.py</code>‏</a>.

</div>

</div>


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from labs.tests import test_lab6

%matplotlib inline
print("تم تهيئة اللاب 6 النهائي بنجاح!")


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:calc(100vw - 140px); max-width:calc(100vw - 140px); min-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">
<style>.markdown, .markdown-body, :host { max-width: 100% !important; width: 100% !important; }</style>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

## ‏2. الاشتقاق الرياضي خطوة بخطوة: من متسلسلة تايلور إلى برهان استحالة المصائد المحلية

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

### ‏أولاً: اشتقاق قالب النقاط الخمس (5-Point Stencil) من متسلسلة تايلور

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏عند أي عقدة داخلية ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(i, j)}$</span>‏ تبعد مسافة ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${h}$</span>‏ عن جيرانها الأربعة:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$\phi_{i+1, j} = \phi_{i,j} + h \left.\frac{\partial\phi}{\partial x}\right|_{i,j} + \frac{h^2}{2}\left.\frac{\partial^2\phi}{\partial x^2}\right|_{i,j} + \frac{h^3}{6}\left.\frac{\partial^3\phi}{\partial x^3}\right|_{i,j} + \mathcal{O}(h^4)$$

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$\phi_{i-1, j} = \phi_{i,j} - h \left.\frac{\partial\phi}{\partial x}\right|_{i,j} + \frac{h^2}{2}\left.\frac{\partial^2\phi}{\partial x^2}\right|_{i,j} - \frac{h^3}{6}\left.\frac{\partial^3\phi}{\partial x^3}\right|_{i,j} + \mathcal{O}(h^4)$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏بجمع المعادلتين معاً، تختفي المشتقات الفردية (‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\partial\phi/\partial x}$</span>‏ و ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\partial^3\phi/\partial x^3}$</span>‏) تماماً:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$\left.\frac{\partial^2\phi}{\partial x^2}\right|_{i,j} \approx \frac{\phi_{i+1, j} - 2\phi_{i,j} + \phi_{i-1, j}}{h^2}, \qquad \left.\frac{\partial^2\phi}{\partial y^2}\right|_{i,j} \approx \frac{\phi_{i, j+1} - 2\phi_{i,j} + \phi_{i, j-1}}{h^2}$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏بالتعويض في معادلة لابلاس ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\frac{\partial^2\phi}{\partial x^2} + \frac{\partial^2\phi}{\partial y^2} = 0}$</span>‏ والضرب في ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${h^2}$</span>‏:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$(\phi_{i+1, j} + \phi_{i-1, j} + \phi_{i, j+1} + \phi_{i, j-1}) - 4\phi_{i,j} = 0 \implies \boxed{\phi_{i,j} = \frac{1}{4}\left(\phi_{i+1, j} + \phi_{i-1, j} + \phi_{i, j+1} + \phi_{i, j-1}\right)}$$

</div>

</div>


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:calc(100vw - 140px); max-width:calc(100vw - 140px); min-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">
<style>.markdown, .markdown-body, :host { max-width: 100% !important; width: 100% !important; }</style>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

### ‏ثانياً: البرهان الرياضي البسيط لاستحالة وجود أي مصيدة محلية (Discrete Maximum Principle)

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏لماذا نضمن بنسبة 100% أن المركبة لن تعلق أبداً في أي حفرة جهد وهمية (Local Minimum)؟

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:14px; line-height:1.85;">

‏• <b>البرهان في سطر واحد:</b> لنفترض جدلاً أن عقدة داخلية حرة ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(i,j)}$</span>‏ هي "قاع محلي صارم" (Strict Local Minimum)، أي أن قيمتها أصغر من جميع جيرانها الأربعة:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$\phi_{i,j} < \phi_{\text{North}}, \quad \phi_{i,j} < \phi_{\text{South}}, \quad \phi_{i,j} < \phi_{\text{East}}, \quad \phi_{i,j} < \phi_{\text{West}}$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:14px; line-height:1.85;">

‏• بجمع المتباينات الأربع والقسمة على 4 نحصل على:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$\phi_{i,j} < \frac{1}{4}\left(\phi_{\text{North}} + \phi_{\text{South}} + \phi_{\text{East}} + \phi_{\text{West}}\right)$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:14px; line-height:1.85;">

‏• ولكن معادلة لابلاس تفرض المساواة التامة ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\phi_{i,j} = \frac{1}{4}(\phi_N + \phi_S + \phi_E + \phi_W)}$</span>‏! <b>تناقض رياضي مستحيل!</b>

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:14px; line-height:1.85;">

‏• إذن: <b>لا يمكن لأي عقدة داخلية حرة أن تكون قاعاً محلياً</b>؛ القاع الوحيد على الخريطة بأكملها هو نقطة الهدف نفسها (‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\phi = 0}$</span>‏)!

</div>

</div>


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:calc(100vw - 140px); max-width:calc(100vw - 140px); min-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">
<style>.markdown, .markdown-body, :host { max-width: 100% !important; width: 100% !important; }</style>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

### ‏ثالثاً: كيف نطبق شرط نيومان العازل (‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\frac{\partial\phi}{\partial n} = 0}$</span>‏) على جدران الحفر؟

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏إذا كان أحد الجيران الأربعة صخرة أو عائقاً (‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">free == False</code>‏)، فلا يجب أن نأخذ قيمته في المتوسط! بدلاً من القسمة على 4 دائماً، نجمع <b>فقط الجيران الحرة المتاحة</b> (‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">pf = phi &#42; f</code>‏) ونقسم على <b>عدد الجيران الحرة الفعلي</b> ‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">cnt</code>‏ (والذي قد يكون 3 بمحاذاة جدار مستقيم، أو 2 في زاوية):

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$\phi_{i,j} = \frac{\sum_{\text{free neighbors}} \phi_{\text{neighbor}}}{\text{cnt}_{i,j}}$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏وهذا يكافئ رياضياً جعل المشتقة العمودية على الجدار تساوي صفراً (‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\frac{\partial\phi}{\partial n} = 0}$</span>‏).

</div>

</div>


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:calc(100vw - 140px); max-width:calc(100vw - 140px); min-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">
<style>.markdown, .markdown-body, :host { max-width: 100% !important; width: 100% !important; }</style>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

## ‏3. 📝 تحدي الورقة والقلم 1 (الحساب اليدوي للتكرار الأول بطريقة جاكوبي — Jacobi Sweep 1)

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏لكي تشعر بما يفعله الكود في كل تكرار، لنحسب يدوياً <b>المسحة الأولى بطريقة جاكوبي المتزامنة (Jacobi Sweep 1)</b> على شبكة صغيرة ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${4 \times 4}$</span>‏ تحتوي على 4 خلايا داخلية ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(A, B, C, D)}$</span>‏ مهيأة في البداية بالقيمة ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\phi^{(0)} = 0.0}$</span>‏:

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; background-color:rgba(128,128,128,0.12); padding:10px 14px; border-radius:6px; overflow-x:auto; margin:10px 0; font-family:monospace; white-space:pre;">‏            الحد العلوي (ثابت: phi = 1.0)
         +--------+--------+--------+--------+
         |  1.0   |  1.0   |  1.0   |  1.0   |
         +--------+--------+--------+--------+
الحد     |  0.5   | A(0.0) | B(0.0) |  0.5   | الحد الأيمن
الأيسر   +--------+--------+--------+--------+ (ثابت Dirichlet:
(ثابت:   |  0.5   | C(0.0) | D(0.0) |  0.5   |  phi = 0.5)
phi=0.5) +--------+--------+--------+--------+
         |  0.0   |  0.0   |  0.0   |  0.0   |
         +--------+--------+--------+--------+
            الحد السفلي (ثابت: phi = 0.0)</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; border-right:4px solid #f59e0b; background-color:rgba(245,158,11,0.08); padding:10px 14px; margin:12px 0; border-radius:4px; line-height:1.85;">

‏📌 <b>ملاحظة مهمة:</b> في طريقة <b>جاكوبي (Jacobi)</b>، نحسب جميع القيم الجديدة ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\phi^{(1)}}$</span>‏ باستخدام القيم القديمة ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\phi^{(0)} = 0.0}$</span>‏ فقط:<br>
‏* جيران الخلية ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${A}$</span>‏ في البداية هم: أعلى ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${= 1.0}$</span>‏، أسفل (‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${C^{(0)}}$</span>‏) ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${= 0.0}$</span>‏، يسار ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${= 0.5}$</span>‏، يمين (‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${B^{(0)}}$</span>‏) ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${= 0.0}$</span>‏.<br>
‏* جيران الخلية ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${B}$</span>‏ في البداية هم: أعلى ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${= 1.0}$</span>‏، أسفل (‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${D^{(0)}}$</span>‏) ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${= 0.0}$</span>‏، يسار (‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${A^{(0)}}$</span>‏) ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${= 0.0}$</span>‏، يمين ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${= 0.5}$</span>‏.<br>
‏<i>(معلومة إضافية: لو كنا نستخدم طريقة Gauss-Seidel المتسلسلة، لكنا استخدمنا قيمة ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${A^{(1)} = 0.375}$</span>‏ الجديدة فوراً عند حساب ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${B^{(1)}}$</span>‏ فتعطي ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${0.46875}$</span>‏؛ لكن هنا نطلب تحديث <b>جاكوبي المتزامن</b>).</i>

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏احسب ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\phi_A^{(1)}}$</span>‏ و ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\phi_B^{(1)}}$</span>‏ بطريقة جاكوبي وأدخلهما أدناه:

</div>

<details dir="rtl" style="direction:rtl; text-align:right; margin:8px 0; padding:6px 10px;">

<summary dir="rtl" style="direction:rtl; text-align:right; cursor:pointer; font-weight:bold;">

‏💡 <b>تلميح (اضغط هنا إذا احتجت مساعدة)</b>

</summary>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏اجمع الجيران الأربعة الابتدائيين للخلية ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${A}$</span>‏: ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(1.0 + 0.0 + 0.5 + 0.0) / 4 = 1.5 / 4}$</span>‏. وبسبب التماثل الأفقي للشبكة، فإن الخلية ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${B}$</span>‏ لها نفس مجموع الجيران الابتدائيين تماماً!

</div>

</details>

</div>


In [ ]:
# خلية تقييم: حساب الشبكة اليدوي (Jacobi Sweep 1)
### START CODE HERE ###
phi_A = 0.375
phi_B = 0.375
### END CODE HERE ###

# تشغيل اختبار التحقق التلقائي:
test_lab6.check_paper_challenge_1(phi_A, phi_B)


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:calc(100vw - 140px); max-width:calc(100vw - 140px); min-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">
<style>.markdown, .markdown-body, :host { max-width: 100% !important; width: 100% !important; }</style>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

## ‏4. 🚀 تحدي الورقة والقلم 2 (معامل الاسترخاء الأمثل ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\omega_{\text{opt}}}$</span>‏ — الدالة ‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">optimal&#95;omega</code>‏)

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏طريقة جاكوبي بطيئة جداً على الشبكات الكبيرة لأن المعلومات تنتقل بمعدل خلية واحدة في كل تكرار.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏لتسريع الحل عشرات المرات، نستخدم <b>الاسترخاء الفائق المتتالي (Successive Over-Relaxation - SOR)</b>، الذي يأخذ خطوة جريئة في اتجاه التغيير بمعامل تضخيم ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\omega \in (1, 2)}$</span>‏:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$\phi_{i,j}^{\text{new}} = (1 - \omega)\,\phi_{i,j}^{\text{old}} + \omega\,\text{avg}_{i,j}$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏وفقاً لنظرية يونغ (Young's Theorem) المطبقة في دالة <a href="rover_path_finder/solvers.py">‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">optimal&#95;omega(N)</code>‏</a> في ملف ‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">solvers.py</code>‏، يُعطى المعامل الأمثل لشبكة حجمها ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${N}$</span>‏ بالعلاقة:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$\omega_{\text{opt}} = \frac{2}{1 + \sin\left(\frac{\pi}{N}\right)}$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏احسب قيمة ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\omega_{\text{opt}}}$</span>‏ عندما يكون ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${N = 16}$</span>‏:

</div>

<details dir="rtl" style="direction:rtl; text-align:right; margin:8px 0; padding:6px 10px;">

<summary dir="rtl" style="direction:rtl; text-align:right; cursor:pointer; font-weight:bold;">

‏💡 <b>تلميح (اضغط هنا إذا احتجت مساعدة)</b>

</summary>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏عوض بـ ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${N = 16}$</span>‏ في القانون ‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">2.0 / (1.0 + np.sin(np.pi / 16))</code>‏ (أو احسبها بالآلة الحاسبة بوضع الراديان: ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\sin(\pi/16) \approx 0.19509 \implies \omega_{\text{opt}} \approx 1.6735}$</span>‏).

</div>

</details>

</div>


In [ ]:
# خلية تقييم: معامل أوميجا الأمثل لـ N=16
### START CODE HERE ###
omega_16 = 2.0 / (1.0 + np.sin(np.pi / 16))
### END CODE HERE ###

# تشغيل اختبار التحقق التلقائي:
test_lab6.check_paper_challenge_2(omega_16)


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:calc(100vw - 140px); max-width:calc(100vw - 140px); min-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">
<style>.markdown, .markdown-body, :host { max-width: 100% !important; width: 100% !important; }</style>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

## ‏5. 💻 التطبيق البرمجي ببايثون: بناء محرك ‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">sor&#95;neumann&#95;step</code>‏ الفعلي

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏الآن ستكتب بنفسك القلب النابض لملف <a href="rover_path_finder/solvers.py">‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">rover&#95;path&#95;finder/solvers.py</code>‏</a>: دالة ‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">sor&#95;neumann&#95;step</code>‏ التي تجمع بين:

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:7px 0; padding-right:8px; line-height:1.85;">

‏<b>1.</b> <b>تقسيم رقعة الشطرنج (Red-Black Ordering):</b> تحديث الخلايا الحمراء (‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">(x + y) % 2 == 0</code>‏) دفعة واحدة باستخدام متجهات NumPy السريعة، ثم تحديث الخلايا السوداء (‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">(x + y) % 2 == 1</code>‏) بالاعتماد على القيم الحمراء المحدثة للتو!

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:7px 0; padding-right:8px; line-height:1.85;">

‏<b>2.</b> <b>شرط نيومان العازل (‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">pf = phi &#42; f</code>‏ والقسمة على ‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">cnt</code>‏):</b> لضمان عدم تسرب الجهد داخل العوائق.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:7px 0; padding-right:8px; line-height:1.85;">

‏<b>3.</b> <b>تحديث الاسترخاء الفائق (SOR):</b> ‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">phi[mk] = (1 - omega) &#42; old + omega &#42; avg[mk]</code>‏.

</div>

</div>


In [ ]:
# دالة تقييم برمجية: sor_neumann_step
def sor_neumann_step(phi, fixed, free, omega=1.5, tol=1e-5, maxit=5000):
    # خوارزمية Red-Black SOR مع جدران نيومان العازلة للعوائق (مطابقة لـ solvers.py)
    phi = phi.copy()
    n, m = phi.shape
    f = free.astype(float)
    
    # حساب عدد الجيران الحرة (من 1 إلى 4) لكل خلية لتطبيق شرط نيومان d(phi)/dn = 0
    cnt = sum(np.roll(f, s, a) for a in (0, 1) for s in (1, -1))
    cnt = np.maximum(cnt, 1.0)
    
    yy, xx = np.mgrid[0:n, 0:m]
    parity = (xx + yy) % 2
    masks = [(parity == c) & free & ~fixed for c in (0, 1)]
    phi[~free] = 0.0
    hist = []
    
    ### START CODE HERE ###
    for _ in range(maxit):
        d = 0.0
        for mk in masks:
            pf = phi * f
            avg = (
                np.roll(pf, 1, 0) + np.roll(pf, -1, 0)
                + np.roll(pf, 1, 1) + np.roll(pf, -1, 1)
            ) / cnt
            old = phi[mk]
            phi[mk] = (1 - omega) * old + omega * avg[mk]
            d = max(d, float(np.abs(phi[mk] - old).max()))
        hist.append(d)
        if d < tol:
            break
    ### END CODE HERE ###
    
    return phi, hist

# تشغيل الاختبار الآلي:
test_lab6.test_sor_neumann_step(sor_neumann_step)


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:calc(100vw - 140px); max-width:calc(100vw - 140px); min-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">
<style>.markdown, .markdown-body, :host { max-width: 100% !important; width: 100% !important; }</style>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

## ‏6. 🌉 لماذا تتفوق معادلة لابلاس على طريقة المجالات الاصطناعية الكلاسيكية (APF)؟

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏في الروبوتات التقليدية، تستخدم طريقة <b>مجال الجهد الاصطناعي (Artificial Potential Field - APF)</b> الموجودة في <a href="rover_path_finder/planners.py">‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">planners.apf&#95;potential</code>‏</a> جمعاً جبرياً بسيطاً لجهد جاذب نحو الهدف وجهد طارد من الصخور:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$U(\mathbf{x}) = U_{\text{att}}(\mathbf{x}) + U_{\text{rep}}(\mathbf{x})$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏<b>أين تكمن الكارثة الهندسية في APF؟</b>

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:14px; line-height:1.85;">

‏• عندما تواجه المركبة عائقاً مقعراً أو حفرة على شكل حرف <b>U</b>، تسحب قوة الجذب المركبة للأمام نحو الهدف خلف الجدار، بينما تدفعها قوة التنافر للخلف بعيداً عن الجدار.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:14px; line-height:1.85;">

‏• عند نقطة داخل الحفرة، <b>تتساوى القوتان في المقدار وتتعاكسان في الاتجاه</b> (‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\nabla U = \mathbf{0}}$</span>‏)، فتتكون <b>مصيدة قاع محلي (Local Minimum Trap)</b> لأن ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\nabla^2 U \neq 0}$</span>‏!

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:14px; line-height:1.85;">

‏• أما في <b>معادلة لابلاس (‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\nabla^2\phi = 0}$</span>‏)</b> مع شروط نيومان العازلة (‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">neumann&#95;potential</code>‏)، فقد أثبتنا رياضياً بمبدأ القيمة العظمى استحالة وجود أي قاع محلي داخل المجال؛ لذلك يلتف تيار الجهد بسلاسة حول جدار الحفرة ويقود المركبة بأمان تام إلى الهدف!

</div>

</div>


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:calc(100vw - 140px); max-width:calc(100vw - 140px); min-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">
<style>.markdown, .markdown-body, :host { max-width: 100% !important; width: 100% !important; }</style>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

## ‏7. 🎨 الإطلاق النهائي للمهمة: الملاحة الذاتية على تضاريس المريخ (‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">100x100</code>‏)

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏شغّل الخلية التالية لتوليد التضاريس المريخية وحل معادلة لابلاس واستخراج مسار المركبة (لاحظ كيف يتفادى المسار السماوي جميع مناطق الخطر الحمراء بفضل توافق الإحداثيات ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(x, y)}$</span>‏ مع ‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">descend</code>‏):

</div>

</div>


In [ ]:
from rover_path_finder.terrain import make_terrain
from rover_path_finder.planners import neumann_potential, descend, apf_potential, trap_map

# 1. توليد تضاريس مريخية 100x100 (10,000 عقدة) مع توسيع العوائق بنصف قطر المركبة
t_data = make_terrain(N=100, seed=42, rover_radius=3)
elev, raw_hazards, hazards = t_data["h"], t_data["raw"], t_data["hazard"]
start = t_data["start"]
goal = t_data["goal"]

# 2. حل معادلة لابلاس بشروط نيومان العازلة (tol=1e-6 لتنفيذ سريع ودقيق)
phi, sweeps, elapsed = neumann_potential(hazards, start=start, goal=goal, tol=1e-6)

# 3. تتبع مسار الانحدار مع خطوط الانسياب (معادلة لاجرانج من اللاب 2)
path, reached = descend(phi, hazards, start=start, goal=goal)

# 4. رسم حقل الجهد والمسار الفعلي (p[0] هو x=col، و p[1] هو y=row)
fig, ax = plt.subplots(figsize=(9.5, 7.5))
cs = ax.contourf(phi, levels=30, cmap='viridis')
plt.colorbar(cs, ax=ax, label=r"Harmonic Potential $\phi(x, y)$")
ax.contour(hazards, levels=[0.5], colors='red', linewidths=1.5)

px = [p[0] for p in path]
py = [p[1] for p in path]
ax.plot(px, py, color='cyan', linewidth=3, label="Rover Trajectory (Laplace Streamline)")
ax.plot(start[0], start[1], 'go', markersize=10, label=r"Start ($\phi=1$)")
ax.plot(goal[0], goal[1], 'r*', markersize=14, label=r"Goal ($\phi=0$)")

ax.set_title("Ares Rover Navigation: Laplace Streamlines Around Martian Hazards", fontsize=13)
ax.set_xlabel("x (column index)")
ax.set_ylabel("y (row index)")
ax.legend(loc='upper left')
plt.show()

print(f"✅ حالة المهمة: وصلت المركبة للهدف = {reached} | طول المسار = {len(path)} خطوة | عدد مسحات SOR = {sweeps} ({elapsed:.2f} ثانية)")


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:calc(100vw - 140px); max-width:calc(100vw - 140px); min-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">
<style>.markdown, .markdown-body, :host { max-width: 100% !important; width: 100% !important; }</style>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

## ‏8. 🪤 الاختبار الحاسم: مقارنة مصيدة حرف U (APF مقابل معادلة لابلاس)

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏شغّل الخلية التالية لمشاهدة المقارنة المباشرة على خريطة المصيدة ‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">trap&#95;map(N=100)</code>‏:

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:14px; line-height:1.85;">

‏• <b>على اليسار (APF):</b> تدخل المركبة من ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(20, 50)}$</span>‏ باتجاه الهدف ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(90, 50)}$</span>‏ فتعلق للأبد داخل جيب حرف U!

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:14px; line-height:1.85;">

‏• <b>على اليمين (Laplace PDE):</b> يلتف المسار السماوي حول جدار المصيدة ويصل إلى الهدف ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(90, 50)}$</span>‏ بنجاح تام!

</div>

</div>


In [ ]:
u_trap, trap_start, trap_goal = trap_map(N=100)

# 1. الروبوت التقليدي (APF)
apf_field = apf_potential(u_trap, goal=trap_goal)
apf_path, apf_reached = descend(apf_field, u_trap, start=trap_start, goal=trap_goal, maxsteps=200)

# 2. معادلة لابلاس التوافقية مع شروط نيومان (tol=1e-6)
laplace_field, trap_sweeps, trap_time = neumann_potential(u_trap, start=trap_start, goal=trap_goal, tol=1e-6)
laplace_path, laplace_reached = descend(laplace_field, u_trap, start=trap_start, goal=trap_goal, maxsteps=400)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# رسم الروبوت الكلاسيكي (APF)
axes[0].contourf(apf_field, levels=25, cmap='magma')
axes[0].contour(u_trap, levels=[0.5], colors='white', linewidths=1.5)
axes[0].plot([p[0] for p in apf_path], [p[1] for p in apf_path], 'r-', linewidth=3, label="APF Path (Stuck)")
axes[0].plot(trap_start[0], trap_start[1], 'go', markersize=9, label="Start (20, 50)")
axes[0].plot(trap_goal[0], trap_goal[1], 'y*', markersize=13, label="Goal (90, 50)")
axes[0].set_title("Classical Robotics (APF): TRAPPED in Local Minimum", fontsize=12)
axes[0].set_xlabel("x")
axes[0].set_ylabel("y")
axes[0].legend(loc="upper left")

# رسم معادلة لابلاس (Laplace PDE)
axes[1].contourf(laplace_field, levels=25, cmap='viridis')
axes[1].contour(u_trap, levels=[0.5], colors='red', linewidths=1.5)
axes[1].plot([p[0] for p in laplace_path], [p[1] for p in laplace_path], 'c-', linewidth=3, label="Laplace Path (Escapes)")
axes[1].plot(trap_start[0], trap_start[1], 'go', markersize=9, label="Start (20, 50)")
axes[1].plot(trap_goal[0], trap_goal[1], 'y*', markersize=13, label="Goal (90, 50)")
axes[1].set_title("Laplace PDE: Maneuvers Cleanly Around the U-Trap", fontsize=12)
axes[1].set_xlabel("x")
axes[1].set_ylabel("y")
axes[1].legend(loc="upper left")

plt.tight_layout()
plt.show()

print("نتائج المقارنة الحاسمة:")
print(f"  • الروبوت التقليدي (APF) : هل وصل للهدف؟ {apf_reached} (علق داخل المصيدة المقعرة)")
print(f"  • معادلة لابلاس (Laplace) : هل وصل للهدف؟ {laplace_reached} (التف حول الجدار ووصل للهدف في {len(laplace_path)} خطوة و {trap_time:.2f} ثانية!)")


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:calc(100vw - 140px); max-width:calc(100vw - 140px); min-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">
<style>.markdown, .markdown-body, :host { max-width: 100% !important; width: 100% !important; }</style>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

## ‏🏆 شهادة إتمام الكورس التطبيقي للمعادلات التفاضلية الجزئية

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏<b>تهانينا الحارة على إتمام اللابات الستة بنجاح!</b>

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏لقد بنيت بيدك الجسر الكامل بين <b>مذكرة المعادلات التفاضلية الجزئية</b> و<b>محرك ملاحة الفضاء ‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">PDE-Rover-Engine</code>‏</b>:

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:7px 0; padding-right:8px; line-height:1.85;">

‏<b>1.</b> <b>اللاب 1 (حذف الثوابت الاختيارية):</b> فهمت معنى المشتقتين الجزئيتين ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(p, q)}$</span>‏ وربطتهما بحساب خريطة انحدار التضاريس <a href="rover_path_finder/terrain.py">‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">terrain.slope&#95;map</code>‏</a>.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:7px 0; padding-right:8px; line-height:1.85;">

‏<b>2.</b> <b>اللاب 2 (طريقة لاجرانج ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${Pp+Qq=R}$</span>‏):</b> اشتققت المنحنيات المميزة وربطتها بمتتبع خطوط انسياب الحركة <a href="rover_path_finder/planners.py">‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">planners.descend</code>‏</a>.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:7px 0; padding-right:8px; line-height:1.85;">

‏<b>3.</b> <b>اللاب 3 (المعادلات غير الخطية وطريقة شاربيت):</b> حللت معادلة المسافة ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${p^2+q^2=1}$</span>‏ وربطتها بتوسيع العوائق بنصف قطر الأمان <a href="rover_path_finder/terrain.py">‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">terrain.dilate</code>‏</a>.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:7px 0; padding-right:8px; line-height:1.85;">

‏<b>4.</b> <b>اللاب 4 (المؤثرات المتجانسة من الرتب العليا):</b> حللت معادلة لابلاس ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(D_x^2+D_y^2)z=0}$</span>‏ وفهمت لماذا تحقق دوالها التوافقية خاصية متوسط الجيران الأربعة <a href="rover_path_finder/solvers.py">‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">solvers.&#95;avg</code>‏</a>.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:7px 0; padding-right:8px; line-height:1.85;">

‏<b>5.</b> <b>اللاب 5 (المؤثرات غير المتجانسة في الرتبة):</b> حللت العوامل الخطية ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(aD_x+bD_y+c)z=0}$</span>‏ وفهمت لماذا يتلاشى انحدار ديريكليه أسياً في الممرات الضيقة.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:7px 0; padding-right:8px; line-height:1.85;">

‏<b>6.</b> <b>اللاب 6 (الفروق المحدودة وخوارزمية SOR):</b> برمجت محرك ‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">sor&#95;neumann&#95;step</code>‏ مع شروط نيومان العازلة وأثبتَّ عملياً كيف تتغلب معادلة لابلاس على مصائد حرف U التي تعجز عنها خوارزميات الروبوتات التقليدية!

</div>

</div>
